# Notebook 01 — ขอบเขตเมืองและการอ่านภาพดาวเทียม

**หัวข้อ:** ขอบเขตเมืองและการอ่านภาพดาวเทียมด้วย Google Earth Engine

**คำถามหลัก:** *เมืองถูกแทนด้วยข้อมูลภูมิศาสตร์อย่างไร และเราสามารถอ่านโครงสร้างเมืองจากภาพดาวเทียมได้อย่างไร?*

### วัตถุประสงค์การเรียนรู้
เมื่อจบ Notebook นี้ นิสิตควรสามารถ:
- อธิบายความแตกต่างระหว่าง **Raster** และ **Vector**
- อธิบาย `pixel`, `spatial resolution`, `temporal resolution` และ `spectral band`
- แยกความแตกต่างระหว่าง `ee.Image`, `ee.ImageCollection`, `ee.Feature`, `ee.FeatureCollection`
- เลือกขอบเขตพื้นที่ศึกษาจากฐานข้อมูลเชิงพื้นที่
- filter ภาพ Sentinel-2 ตามพื้นที่ เวลา และเมฆ
- สร้าง true-color และ false-color composite
- ใช้ interactive map เพื่อสังเกตรูปแบบเชิงพื้นที่ของเมือง
- อธิบายว่าขอบเขตการปกครองไม่จำเป็นต้องเท่ากับขอบเขตเมืองจริง

## 1. แนวคิดพื้นฐาน

### Raster และ Vector

**Vector** เหมาะกับวัตถุที่มีรูปร่างชัด เช่น จุด ถนน และขอบเขตการปกครอง  
**Raster** แทนพื้นที่ด้วยตาราง pixel เช่น ภาพดาวเทียม NDVI และ Land Surface Temperature

ใน Earth Engine:

| แนวคิด GIS | Earth Engine |
|---|---|
| Raster 1 ภาพ | `ee.Image` |
| Raster หลายช่วงเวลา | `ee.ImageCollection` |
| Vector 1 วัตถุ | `ee.Feature` |
| Vector หลายวัตถุ | `ee.FeatureCollection` |

### Administrative city กับ Physical urban area
ขอบเขตจังหวัดหรือกรุงเทพมหานครเป็น **administrative boundary** แต่พื้นที่ก่อสร้างจริงอาจมีขนาดและรูปแบบแตกต่างจากขอบเขตดังกล่าวอย่างมาก

## การเตรียมระบบ (ทำทุก Notebook)

Notebook นี้ออกแบบสำหรับ **Google Colab + Google Earth Engine Python API + geemap**

### ก่อนเริ่ม
1. ต้องมีบัญชี Google Earth Engine ที่เปิดใช้งานแล้ว
2. ต้องมี Google Cloud Project ที่ใช้กับ Earth Engine
3. แก้ค่า `PROJECT` ใน code cell ด้านล่างให้เป็น Project ID ของตนเอง

> **แนวคิดสำคัญ:** Python ใน Colab เป็นฝั่งผู้ใช้ (client) ส่วน `ee.Image`, `ee.ImageCollection` และการคำนวณส่วนใหญ่เป็นวัตถุแบบ server-side ที่ประมวลผลบน Google Earth Engine

In [5]:
%pip install -q earthengine-api geemap pandas matplotlib scikit-learn scipy

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Google Cloud Project ID ที่ลงทะเบียน Earth Engine แล้ว
PROJECT = "ee-apichayaice7835" #<-- นิสิต ก๊อปปี้ของตนเองมาใส่ตรงนี้

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print("Earth Engine initialized successfully.")
print("Using project:", PROJECT)

# -------------------------------------------------------------------
# ฟังก์ชันมาตรฐานสำหรับทุกแผนที่ในรายวิชา
# 1) สร้าง geemap.Map()
# 2) ลบ default OpenStreetMap ที่ถูก block ใน Colab environment นี้
# 3) ใช้ Esri.WorldImagery เป็น basemap เริ่มต้น
# 4) เตรียม Street Map และ Topographic Map ไว้ให้เลือกใน Layer Control
# -------------------------------------------------------------------
def create_course_map():
    m = geemap.Map()
    m.clear_layers()
    m.add_basemap("Esri.WorldImagery", show=True)
    m.add_basemap("Esri.WorldStreetMap", show=False)
    m.add_basemap("Esri.WorldTopoMap", show=False)
    return m

Earth Engine initialized successfully.
Using project: ee-apichayaice7835


### Basemap ที่ใช้ในชุด Notebook นี้

ทุกแผนที่เรียกผ่าน `create_course_map()` เพื่อหลีกเลี่ยง default OpenStreetMap tile ที่ถูก block ใน Colab environment นี้ โดยกำหนด **Esri.WorldImagery** เป็นพื้นหลังเริ่มต้น และมี **Esri.WorldStreetMap** กับ **Esri.WorldTopoMap** ให้เปิดจาก Layer Control ตามวัตถุประสงค์ของการอ่านแผนที่


## 2. เรียกขอบเขตประเทศไทยและตรวจสอบชื่อจังหวัด

ใช้ **FAO GAUL 2025 Level 1**  
field สำคัญ:
- `GAUL0_NAME` = ประเทศ
- `GAUL1_NAME` = หน่วยปกครองระดับที่ 1

ก่อน filter ควรดู attribute จริงก่อนเสมอ

In [6]:
admin1 = ee.FeatureCollection("FAO/GAUL/2025/level1")
thailand = admin1.filter(ee.Filter.eq("GAUL0_NAME", "Thailand"))

province_names = thailand.aggregate_array("GAUL1_NAME").sort().getInfo()
province_names

['Amnat Charoen',
 'Ang Thong',
 'Bangkok',
 'Bueng Kan',
 'Buri Ram',
 'Chachoengsao',
 'Chai Nat',
 'Chaiyaphum',
 'Chanthaburi',
 'Chiang Mai',
 'Chiang Rai',
 'Chon Buri',
 'Chumphon',
 'Kalasin',
 'Kamphaeng Phet',
 'Kanchanaburi',
 'Khon Kaen',
 'Krabi',
 'Lampang',
 'Lamphun',
 'Loei',
 'Lop Buri',
 'Mae Hong Son',
 'Maha Sarakham',
 'Mukdahan',
 'Nakhon Nayok',
 'Nakhon Pathom',
 'Nakhon Phanom',
 'Nakhon Ratchasima',
 'Nakhon Sawan',
 'Nakhon Si Thammarat',
 'Nan',
 'Narathiwat',
 'Nong Bua Lam Phu',
 'Nong Khai',
 'Nonthaburi',
 'Pathum Thani',
 'Pattani',
 'Phangnga',
 'Phatthalung',
 'Phayao',
 'Phetchabun',
 'Phetchaburi',
 'Phichit',
 'Phitsanulok',
 'Phra Nakhon Si Ayutthaya',
 'Phrae',
 'Phuket',
 'Prachin Buri',
 'Prachuap Khiri Khan',
 'Ranong',
 'Ratchaburi',
 'Rayong',
 'Roi Et',
 'Sa Kaeo',
 'Sakon Nakhon',
 'Samut Prakan',
 'Samut Sakhon',
 'Samut Songkhram',
 'Saraburi',
 'Satun',
 'Si Sa Ket',
 'Sing Buri',
 'Songkhla',
 'Sukhothai',
 'Suphan Buri',
 'Surat Than

In [9]:
# Administrative boundary: FAO GAUL 2025 Level 1
admin1 = ee.FeatureCollection("FAO/GAUL/2025/level1")
thailand = admin1.filter(ee.Filter.eq("GAUL0_NAME", "Thailand"))

def get_admin1(name_contains):
    # เลือกหน่วยปกครองระดับ 1 ของประเทศไทยด้วยข้อความบางส่วนใน GAUL1_NAME
    return thailand.filter(
        ee.Filter.stringContains("GAUL1_NAME", name_contains)
    )

bangkok = get_admin1("Bangkok")
chiang_mai = get_admin1("Chiang Mai")
Kalasin = get_admin1("Kalasin")
print("Bangkok features:", bangkok.size().getInfo())
print("Chiang Mai features:", chiang_mai.size().getInfo())
print("Kalasin features:", Kalasin.size().getInfo())

Bangkok features: 1
Chiang Mai features: 1
Kalasin features: 1


## 3. สาธิต: แสดงขอบเขตกรุงเทพมหานคร

การแสดงขอบเขตในแผนที่ช่วยให้นิสิตเข้าใจว่า **FeatureCollection** สามารถใช้เป็นทั้งข้อมูลสำหรับวิเคราะห์และ layer สำหรับอ้างอิงเชิงพื้นที่

In [12]:
Map = create_course_map()
# จัดตำแหน่งแผนที่
Map.centerObject(Kalasin, 9)

# ขอบเขตกรุงเทพมหานคร
Kalasin_outline = Kalasin.style(
    color="red",
    fillColor="00000000",
    width=3
)

Map.addLayer(
    Kalasin_outline,
    {},
    "Kalasin administrative boundary"
)

# ให้ผู้ใช้เปิด/ปิดและเลือก layer ได้
Map.add_layer_control()

Map

Map(center=[16.628106258151956, 103.6218636219392], controls=(WidgetControl(options=['position', 'transparent_…

## 4. Sentinel-2 Surface Reflectance

ใช้ dataset:

`COPERNICUS/S2_SR_HARMONIZED`

Band ที่ใช้ในคาบนี้:
- B2 = Blue, 10 m
- B3 = Green, 10 m
- B4 = Red, 10 m
- B8 = Near Infrared (NIR), 10 m

เราจะสร้าง composite ช่วงฤดูค่อนข้างแห้ง เพื่อลดผลของเมฆ และใช้ **median composite** เพื่อลดอิทธิพลของ observation ที่ผิดปกติ

> หมายเหตุ: cloud masking ในที่นี้ใช้ `SCL` เพื่อให้โค้ดอ่านง่ายสำหรับผู้เริ่มต้น

In [15]:
def mask_s2_sr(image):
    # Mask cloud/shadow classes from Sentinel-2 SCL and convert reflectance to 0-1.
    scl = image.select("SCL")
    clear = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    scaled = image.select("B.*").multiply(0.0001)
    return (
        scaled.updateMask(clear)
        .copyProperties(image, ["system:time_start"])
    )

def sentinel2_composite(region, start_date, end_date, max_cloud=40):
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud))
        .map(mask_s2_sr)
    )
    composite = collection.median().clip(region)
    return collection, composite

START = "2024-11-01"
END   = "2025-03-01"

s2_bkk, bkk_rgb = sentinel2_composite(Kalasin.geometry(), START, END)

print("จำนวน Sentinel-2 scenes:", s2_bkk.size().getInfo())

จำนวน Sentinel-2 scenes: 139


## 5. แสดง True Color และ False Color

True color: B4, B3, B2  
False color สำหรับเน้น vegetation: B8, B4, B3

ให้เปิด-ปิด layer สลับกัน และสังเกตแม่น้ำ พื้นที่ก่อสร้าง vegetation และพื้นที่ชายขอบเมือง

In [24]:
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0.02,
    "max": 0.30,
    "gamma": 1.1,
}

false_vis = {
    "bands": ["B8", "B4", "B3"],
    "min": 0.02,
    "max": 0.35,
    "gamma": 1.1,
}

Agi_vis = {
    "bands": ["B12", "B11", "B8"],
    "min": 0.02,
    "max": 0.45,
    "gamma": 1.1,
}
Map = create_course_map()
Map.centerObject(Kalasin, 9)

Map.addLayer(bkk_rgb, rgb_vis, "Sentinel-2 True color", False)
Map.addLayer(bkk_rgb, false_vis, "Sentinel-2 False color", True)
# เพิ่มสี AGI
Map.addLayer(bkk_rgb, Agi_vis, "Sentinel-2 Agriculture color", True)
Map.addLayer(Kalasin_outline, {}, "Kalasin boundary")

Map.add_layer_control()
Map

Map(center=[16.628106258151853, 103.6218636219392], controls=(WidgetControl(options=['position', 'transparent_…

### Observe the Map — อ่านแผนที่ก่อนคำนวณ

ตอบคำถามสั้น ๆ:
1. แนวแม่น้ำเจ้าพระยาเห็นได้อย่างไรใน true-color image?
2. บริเวณใดดูเป็น built-up area หนาแน่น?
3. บริเวณใดมี vegetation มาก?
4. รูปแบบเมืองสอดคล้องกับขอบเขตการปกครองทั้งหมดหรือไม่?

## 6. แบบฝึกหัด: Chiang Mai

ให้ทำ workflow เดียวกันกับเชียงใหม่

**งานที่ต้องทำ**
1. แสดงขอบเขตเชียงใหม่
2. สร้าง Sentinel-2 composite ช่วงเดียวกับกรุงเทพฯ
3. แสดง True color
4. แสดง False color
5. อ่านแผนที่และตอบคำถาม

เติมส่วน `TODO` ด้านล่าง

In [ ]:
# TODO 1: กำหนด region
cm_region = None

# TODO 2: สร้าง composite
# s2_cm, cm_rgb = sentinel2_composite(...)

# TODO 3: สร้าง Map และ addLayer
# Map_cm = create_course_map()
# ...

### คำถามการตีความ
1. พื้นที่ built-up หลักของเชียงใหม่อยู่บริเวณใด?
2. ภูเขาทางด้านตะวันตกมีลักษณะใน true color/false color อย่างไร?
3. จังหวัดเชียงใหม่ทั้งหมดควรถูกเรียกว่า “พื้นที่เมืองเชียงใหม่” หรือไม่ เพราะเหตุใด?
4. spatial resolution 10 m เหมาะกับการมองเห็นวัตถุระดับใด?

## 7. สิ่งที่ควรได้จาก Notebook 01

- เข้าใจโครงสร้างข้อมูลหลักของ Earth Engine
- เข้าใจว่าการเลือก **WHERE + WHEN + WHAT** เป็นพื้นฐานของ geospatial analysis
- สามารถอ่านภาพดาวเทียมเบื้องต้น
- เห็นความแตกต่างระหว่าง administrative boundary และ physical urban extent

# SOLUTION — เปิดหลังทำแบบฝึกหัดแล้ว

In [ ]:
cm_region = chiang_mai.geometry()

s2_cm, cm_rgb = sentinel2_composite(cm_region, START, END)

print("จำนวน Sentinel-2 scenes ของเชียงใหม่:", s2_cm.size().getInfo())

cm_outline = chiang_mai.style(
    color="202020",
    fillColor="00000000",
    width=3
)

Map_cm = create_course_map()
Map_cm.centerObject(chiang_mai, 8)

Map_cm.addLayer(cm_rgb, rgb_vis, "Chiang Mai: True color")
Map_cm.addLayer(cm_rgb, false_vis, "Chiang Mai: False color", False)
Map_cm.addLayer(cm_outline, {}, "Chiang Mai boundary")

Map_cm.add_layer_control()
Map_cm

จำนวน Sentinel-2 scenes ของเชียงใหม่: 276


Map(center=[18.78744159805183, 98.72513681282037], controls=(WidgetControl(options=['position', 'transparent_b…

### แนวคำตอบ
- พื้นที่ก่อสร้างหลักจะกระจุกในแอ่งเชียงใหม่บริเวณเมืองเชียงใหม่และพื้นที่ต่อเนื่องโดยรอบ
- ทางตะวันตกปรากฏภูมิประเทศสูงและ vegetation เด่นชัด
- ขอบเขตจังหวัดมีพื้นที่ภูเขา ป่า เกษตร และชนบทจำนวนมาก จึงไม่ใช่ขอบเขต physical city
- Sentinel-2 10 m เหมาะกับ urban block, ถนนสายใหญ่, แหล่งน้ำ และพื้นที่สีเขียวขนาดกลางขึ้นไป มากกว่าการระบุอาคารเดี่ยวอย่างน่าเชื่อถือ

## แหล่งข้อมูลอ้างอิง
- FAO GAUL 2025 Level 1: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2025_level1
- Sentinel-2 SR Harmonized: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
- Earth Engine Python tutorial: https://developers.google.com/earth-engine/tutorials/community/intro-to-python-api